# WAXAL-NET Results Aggregation
Reads all `metrics.json` files from `/kaggle/working/outputs/` and assembles
the paper's result tables:
- **Table 1**: Zero-shot results (WER% / CER%) for all 19 languages × 3 zero-shot models
- **Table 3**: Fine-tuning results (WER% / CER%) for all 19 languages × 3 fine-tuned models

Outputs are saved as Markdown tables and a combined CSV.

In [ ]:
import json, os
from pathlib import Path
import csv

In [ ]:
LANGUAGES = ["acholi", "akan", "amharic", "dagaare", "dagbani", "ewe", "fula", "ikposo",
             "lingala", "luganda", "malagasy", "masaaba", "nyankole", "oromo", "shona",
             "sidama", "soga", "tigrinya", "wolaytta"]
ZS_MODELS = ["omnilingual_1b", "mms_1b", "whisper_large_v3"]
FT_MODELS = ["mms_300m", "whisper_small", "whisper_tiny"]
OUTPUTS = Path("/kaggle/working/outputs")

def get_wer(run_name):
    p = OUTPUTS / run_name / "metrics.json"
    if not p.exists():
        return None, None
    m = json.loads(p.read_text())
    if m.get("status") == "N/A":
        return "N/A", "N/A"
    return round(m["wer_percent"], 1), round(m["cer_percent"], 1)

In [ ]:
# Table 1: Zero-shot results
header = "| Language | Omni-1B (ZS) | MMS-1B (ZS) | Whisper-L (ZS) |"
sep = "|---|---|---|---|"
rows = [header, sep]
zs_wers = {m: [] for m in ZS_MODELS}
for lang in LANGUAGES:
    cells = [lang.title()]
    for model in ZS_MODELS:
        wer, cer = get_wer(f"zero_shot_{lang}_{model}")
        if wer is None:
            cells.append("\u2014")
        elif wer == "N/A":
            cells.append("N/A")
        else:
            cells.append(f"{wer} / {cer}")
            zs_wers[model].append(wer)
    rows.append("| " + " | ".join(cells) + " |")
# Macro average row
avg_cells = ["**Macro Avg**"]
for model in ZS_MODELS:
    if zs_wers[model]:
        avg_cells.append(f"**{sum(zs_wers[model])/len(zs_wers[model]):.1f}**")
    else:
        avg_cells.append("\u2014")
rows.append("| " + " | ".join(avg_cells) + " |")
table1_md = "\n".join(rows)
print("## Table 1: Zero-shot Results (WER% / CER%)")
print(table1_md)

In [ ]:
# Table 3: Fine-tuning results
header3 = "| Language | MMS-300M (FT) | Whisper-S (FT) | Whisper-T (FT) |"
sep3 = "|---|---|---|---|"
rows3 = [header3, sep3]
ft_wers = {m: [] for m in FT_MODELS}
for lang in LANGUAGES:
    cells = [lang.title()]
    for model in FT_MODELS:
        wer, cer = get_wer(f"finetune_{lang}_{model}")
        if wer is None:
            cells.append("\u2014")
        elif wer == "N/A":
            cells.append("N/A")
        else:
            cells.append(f"{wer} / {cer}")
            ft_wers[model].append(wer)
    rows3.append("| " + " | ".join(cells) + " |")
# Macro average row
avg_cells3 = ["**Macro Avg**"]
for model in FT_MODELS:
    if ft_wers[model]:
        avg_cells3.append(f"**{sum(ft_wers[model])/len(ft_wers[model]):.1f}**")
    else:
        avg_cells3.append("\u2014")
rows3.append("| " + " | ".join(avg_cells3) + " |")
table3_md = "\n".join(rows3)
print("## Table 3: Fine-tuning Results (WER% / CER%)")
print(table3_md)

In [ ]:
out = Path("/kaggle/working/outputs")
out.mkdir(exist_ok=True)
(out / "table1_zeroshot.md").write_text(table1_md)
(out / "table3_finetune.md").write_text(table3_md)
# Also save CSV
with open(out / "waxal_results.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["language", "stage", "model", "wer_percent", "cer_percent"])
    for lang in LANGUAGES:
        for model in ZS_MODELS:
            wer, cer = get_wer(f"zero_shot_{lang}_{model}")
            writer.writerow([lang, "zero_shot", model, wer, cer])
        for model in FT_MODELS:
            wer, cer = get_wer(f"finetune_{lang}_{model}")
            writer.writerow([lang, "finetune", model, wer, cer])
print("Results saved to /kaggle/working/outputs/")